# Ch.1 — Clustering

> **Running theme:** The retail company wants to discover natural customer segments from purchase behaviour alone — no labels, no target variable. K-Means, DBSCAN, and HDBSCAN each partition the data differently. Together they reveal the hidden structure in 440 wholesale customers.

## Core Idea

**K-Means:** minimise inertia $J = \sum_k \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$ via alternating assignment → centroid-update steps (Lloyd's algorithm).

**DBSCAN:** density-based — a core point has ≥ `min_samples` neighbours within radius `ε`. Clusters = maximal density-connected sets. Outliers labelled **−1** (noise).

**HDBSCAN:** builds a hierarchy over all density levels via mutual reachability distance + MST. Extracts the most stable clusters. Only needs `min_cluster_size`.

## Running Example

Dataset: **UCI Wholesale Customers** — 440 customers, 6 spending features.
**No target variable** — pure unsupervised learning.

Question: *What natural customer types exist?* After clustering we colour customers by K-Means cluster label to see if spending tiers, channel splits (hotel vs retail), or speciality patterns emerge.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from pathlib import Path

IMG = Path("img"); IMG.mkdir(exist_ok=True)

# ── Load data ──────────────────────────────────────────────────────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']
X = df[spend_cols].values  # (440, 6)

# Log-transform (spending is heavily right-skewed) + standardise
X_log = np.log1p(X)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_log)

print(f"Dataset: {X.shape[0]} customers × {X.shape[1]} features")
print("Features:", spend_cols)
print(f"\nRaw range example — Fresh: {X[:, 0].min():.0f} to {X[:, 0].max():,.0f}")
print(f"After log+scale  — Fresh: {X_sc[:, 0].min():.2f} to {X_sc[:, 0].max():.2f}")

## K-Means: Elbow Curve

Inertia (within-cluster sum of squares) always decreases as K increases.
The **elbow** — where marginal gain flattens — suggests the best K.
Silhouette score provides a second, independent signal.

In [ ]:
# ── K-Means elbow + silhouette sweep ──────────────────────────────────────────
K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sc, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(K_range), inertias, 'b-o', markersize=5)
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Curve — inertia vs K')
ax1.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(list(K_range), sil_scores, 'r-o', markersize=5)
ax2.set_xlabel('K'); ax2.set_ylabel('Mean silhouette score')
ax2.set_title('Silhouette score vs K')
ax2.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(IMG / "ch01_elbow_silhouette.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

best_k_sil = list(K_range)[np.argmax(sil_scores)]
print(f"Best K by silhouette: {best_k_sil}  (score={max(sil_scores):.4f})")
print(f"Silhouette at K=5: {sil_scores[3]:.4f}")

## K-Means: Fit and Interpret Clusters

Fit with K=5 (business requirement: 5 actionable segments).
Inspect centroids in the **original** (un-scaled) feature space to understand what each segment represents.

In [ ]:
# ── Fit K-Means with K=5 ──────────────────────────────────────────────────────
best_k = 5
km_best = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
km_best.fit(X_sc)
labels_km = km_best.labels_

# Decode centroids to original scale (undo log + scale)
centroids_log = scaler.inverse_transform(km_best.cluster_centers_)
centroids_orig = np.expm1(centroids_log)

segment_names = ["Loyalists", "Price-Sensitive", "Big Spenders",
                 "Occasional Buyers", "Deli Specialists"]

print(f"K = {best_k} customer segments (original spending scale)\n")
for i, c in enumerate(centroids_orig):
    n_pts = (labels_km == i).sum()
    print(f"  Segment {i} — '{segment_names[i]}' (n={n_pts}, {n_pts/len(X)*100:.0f}%):")
    for col, val in zip(spend_cols, c):
        print(f"    {col:<18} {val:>10,.0f}")
    print()

## K-Means: 2D Visualisation via PCA

We can't plot 6 dimensions directly. Use PCA to project to 2D and colour by cluster label.

In [ ]:
# ── PCA 2D projection ─────────────────────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_sc)
print(f"PCA explained variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cluster labels
scatter_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for i in range(best_k):
    mask = labels_km == i
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=scatter_colors[i],
                    s=15, alpha=0.6, label=segment_names[i])
axes[0].set_title(f'K-Means (K={best_k}) — Customer Segments')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8, markerscale=2)

# Channel colour (proxy validation)
channels = df['Channel'].values
sc1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=channels,
                       cmap='coolwarm', s=15, alpha=0.6)
axes[1].set_title('Actual Channel (1=Hotel, 2=Retail)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[1], label='Channel')

plt.tight_layout()
fig.savefig(IMG / "ch01_kmeans_clusters_2d.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## DBSCAN: ε Estimation via k-NN Distance Plot

**Rule of thumb:** `k = 2 × n_features = 12`.
Sort all customers by their distance to the k-th nearest neighbour.
The **knee** of the resulting curve is a good starting ε.

In [ ]:
# ── k-NN distance plot for ε estimation ────────────────────────────────────────
k_nn = 2 * X_sc.shape[1]  # 12
nbrs = NearestNeighbors(n_neighbors=k_nn, algorithm='ball_tree').fit(X_sc)
distances, _ = nbrs.kneighbors(X_sc)
knn_dists = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(knn_dists, lw=1)
ax.set_xlabel('Customers sorted by distance to 12th neighbour')
ax.set_ylabel('Distance (standardised space)')
ax.set_title('k-NN Distance Plot — pick ε at the knee')
ax.axhline(y=1.5, color='r', linestyle='--', label='ε = 1.5 (candidate)')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(IMG / "ch01_knn_distance.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"50th percentile: {np.percentile(knn_dists, 50):.3f}")
print(f"75th percentile: {np.percentile(knn_dists, 75):.3f}")
print(f"90th percentile: {np.percentile(knn_dists, 90):.3f}")

## DBSCAN: Fit and Inspect

In [ ]:
# ── DBSCAN ────────────────────────────────────────────────────────────────────
eps_val = 1.5
min_samp_val = 12  # 2 × n_features

db = DBSCAN(eps=eps_val, min_samples=min_samp_val, algorithm='ball_tree')
db.fit(X_sc)
labels_db = db.labels_

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()
print(f"DBSCAN (eps={eps_val}, min_samples={min_samp_val})")
print(f"  Clusters found : {n_clusters_db}")
print(f"  Noise customers: {n_noise} ({n_noise/len(X_sc)*100:.1f}%)")

# Visualise in 2D PCA space
fig, ax = plt.subplots(figsize=(8, 5))
colours = labels_db.copy().astype(float)
colours[colours == -1] = np.nan
sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=colours, cmap='tab20', s=15, alpha=0.6)
noise_mask = labels_db == -1
ax.scatter(X_2d[noise_mask, 0], X_2d[noise_mask, 1],
           c='black', s=10, alpha=0.3, marker='x', label='Noise')
plt.colorbar(sc, ax=ax, label='Cluster')
ax.set_title(f'DBSCAN — {n_clusters_db} clusters (× = noise customers)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=2)
plt.tight_layout()
fig.savefig(IMG / "ch01_dbscan_clusters.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Who are the noise customers?
if n_noise > 0:
    print(f"\nNoise customer spending profiles (top 5 by total spend):")
    noise_df = df.loc[noise_mask, spend_cols]
    noise_df['Total'] = noise_df.sum(axis=1)
    print(noise_df.nlargest(5, 'Total')[spend_cols].to_string())

## HDBSCAN

In [ ]:
# ── HDBSCAN ───────────────────────────────────────────────────────────────────
try:
    import hdbscan
    hdb = hdbscan.HDBSCAN(min_cluster_size=22, min_samples=5, gen_min_span_tree=False)
    labels_hdb = hdb.fit_predict(X_sc)

    n_clusters_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    n_noise_hdb = (labels_hdb == -1).sum()
    print(f"HDBSCAN: {n_clusters_hdb} clusters, {n_noise_hdb} noise ({n_noise_hdb/len(X_sc)*100:.1f}%)")

    fig, ax = plt.subplots(figsize=(8, 5))
    h_colours = labels_hdb.copy().astype(float)
    h_colours[h_colours == -1] = np.nan
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=h_colours, cmap='tab20', s=15, alpha=0.6)
    nm = labels_hdb == -1
    ax.scatter(X_2d[nm, 0], X_2d[nm, 1], c='black', s=10, alpha=0.3, marker='x', label='Noise')
    ax.set_title(f'HDBSCAN — {n_clusters_hdb} clusters (× = noise)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(markerscale=2)
    plt.tight_layout()
    fig.savefig(IMG / "ch01_hdbscan_clusters.png", dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

except ImportError:
    print("hdbscan not installed — run: pip install hdbscan")

## Hyperparameter Dial: K-Means K Sweep

Visually compare how segment boundaries shift as K increases from 2 to 6.

In [ ]:
# ── K sweep visualisation ─────────────────────────────────────────────────────
Ks = [2, 3, 4, 5, 6]
fig, axes = plt.subplots(1, len(Ks), figsize=(18, 4))

for ax, k in zip(axes, Ks):
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42)
    km_k.fit(X_sc)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=km_k.labels_, cmap='tab10', s=10, alpha=0.5)
    sil_k = silhouette_score(X_sc, km_k.labels_)
    ax.set_title(f'K={k}  sil={sil_k:.2f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('K-Means: how segment boundaries shift with K', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch01_k_sweep.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## Hyperparameter Dial: DBSCAN ε Sweep

In [ ]:
# ── ε sweep visualisation ─────────────────────────────────────────────────────
eps_values = [0.5, 1.0, 1.5, 2.5]
fig, axes = plt.subplots(1, len(eps_values), figsize=(18, 4))

for ax, eps in zip(axes, eps_values):
    db_e = DBSCAN(eps=eps, min_samples=12, algorithm='ball_tree')
    db_e.fit(X_sc)
    lbl = db_e.labels_
    n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_nz = (lbl == -1).sum()
    cols = lbl.copy().astype(float); cols[cols == -1] = np.nan
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=cols, cmap='tab20', s=10, alpha=0.5)
    mask_n = lbl == -1
    ax.scatter(X_2d[mask_n, 0], X_2d[mask_n, 1], c='black', s=5, alpha=0.3, marker='x')
    ax.set_title(f'ε={eps}\n{n_c} clusters, {n_nz/len(X_sc)*100:.0f}% noise')

plt.suptitle('DBSCAN: ε sweep (too small → all noise; too large → one cluster)', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch01_eps_sweep.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## What Can Go Wrong: K-Means on Non-Spherical Data

K-Means assumes clusters are convex and roughly equal-sized. On ring-shaped or elongated distributions it fails while DBSCAN succeeds.

In [ ]:
# ── Synthetic demo: K-Means vs DBSCAN on non-spherical shapes ─────────────────
from sklearn.datasets import make_circles, make_moons

np.random.seed(42)
X_circles, _ = make_circles(n_samples=800, factor=0.5, noise=0.05)
X_moons, _ = make_moons(n_samples=800, noise=0.07)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (Xd, name) in enumerate([(X_circles, 'Circles'), (X_moons, 'Moons')]):
    km_fail = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xd)
    axes[row, 0].scatter(Xd[:, 0], Xd[:, 1], c=km_fail.labels_, cmap='bwr', s=10, alpha=0.7)
    axes[row, 0].set_title(f'K-Means on {name} — WRONG')

    db_ok = DBSCAN(eps=0.2, min_samples=5).fit(Xd)
    axes[row, 1].scatter(Xd[:, 0], Xd[:, 1], c=db_ok.labels_, cmap='bwr', s=10, alpha=0.7)
    axes[row, 1].set_title(f'DBSCAN on {name} — correct')

plt.suptitle('K-Means fails on non-spherical shapes; DBSCAN succeeds', fontsize=13)
plt.tight_layout()
fig.savefig(IMG / "ch01_nonspherical.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## What Can Go Wrong: Unscaled Features

In [ ]:
# ── Scaling matters ───────────────────────────────────────────────────────────
print("Feature ranges (raw spending):")
for name, mn, mx in zip(spend_cols, X.min(axis=0), X.max(axis=0)):
    print(f"  {name:<18} {mn:>8,.0f} … {mx:>10,.0f}")

print("\nFresh range is 3× wider than Delicatessen — unscaled K-Means is dominated by Fresh.")

from sklearn.metrics import adjusted_rand_score
km_raw = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)       # RAW
km_log = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X_sc)    # LOG + SCALED
ari = adjusted_rand_score(km_log.labels_, km_raw.labels_)
print(f"\nARI between raw vs log+scaled K-Means: {ari:.3f}")
print("(ARI=1.0 means identical; ARI≈0 means completely different clusterings)")

## Exercises

1. **DBSCAN noise analysis.** Examine the noise customers identified by DBSCAN. What makes them outliers? Compute their average spending per feature and compare to the cluster centroids. Are they extreme spenders or minimal spenders?

2. **K-Means++ vs random init.** Run `KMeans(n_clusters=5, init='random', n_init=1)` ten times with different seeds and record inertia. Then run `init='k-means++'`. Plot both inertia distributions as histograms. How much more stable is K-Means++?

3. **Segment radar chart.** Create a radar (spider) plot showing the centroid profile for each of the 5 segments. Which features most distinguish "Big Spenders" from "Price-Sensitive"?

In [ ]:
# Exercise 1 — DBSCAN noise analysis
# TODO: your solution here
pass

In [ ]:
# Exercise 2 — K-Means++ vs random init stability
# TODO: your solution here
pass

In [ ]:
# Exercise 3 — Segment radar chart
# TODO: your solution here
pass